In [ ]:
!pip install pytorch_lightning wandb onnxruntime albumentations matplotlib

import os
import glob
import zipfile
import shutil
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import wandb
import matplotlib.pyplot as plt
import random

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configuration
RAW_ROOT = "/content/drive/MyDrive/DataLake/01_raw"
PROCESSED_ROOT = "/content/drive/MyDrive/DataLake/02_processed/"
ARTIFACTS_ROOT = "/content/drive/MyDrive/DataLake/03_artifacts"
MODEL_SAVE_DIR = os.path.join(ARTIFACTS_ROOT, "models")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Local high-speed extraction directory
LOCAL_DATA_DIR = "/content/data"

BATCH_SIZE = 64
IMG_SIZE = 192 # Crucial: 192 evenly divides by 3 -> 64x64 patches
MAX_EPOCHS = 50
LR = 1e-4
PROJECT_NAME = "puzzle33-pretraining"


In [ ]:
# Data Preparation: Extract zip files from RAW_ROOT to LOCAL_DATA_DIR
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

zips = {
    "train": os.path.join(RAW_ROOT, "train.zip"),
    "valid": os.path.join(RAW_ROOT, "valid.zip"),
    "test": os.path.join(RAW_ROOT, "test.zip")
}

for split, zip_path in zips.items():
    # Fallback for validation name variations
    if not os.path.exists(zip_path) and split == "valid":
        for alt_name in ["val.zip", "validation.zip"]:
            alt_path = os.path.join(RAW_ROOT, alt_name)
            if os.path.exists(alt_path):
                zip_path = alt_path
                break
                
    if os.path.exists(zip_path):
        extract_to = os.path.join(LOCAL_DATA_DIR, split)
        # Clean up any existing directory to avoid stale files
        if os.path.exists(extract_to):
            shutil.rmtree(extract_to)
        os.makedirs(extract_to, exist_ok=True)
        
        print(f"Extracting {zip_path} to {extract_to}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"Successfully extracted {split} split.")
    else:
        print(f"Warning: Zip file not found at {zip_path}")


In [ ]:
# Dataset Analysis Logic: Scan extracted directories and print class counts
from collections import Counter

print("--- Dataset Split Analysis ---")
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(LOCAL_DATA_DIR, split)
    if os.path.exists(split_dir):
        image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
        images = []
        for ext in image_extensions:
            images.extend(glob.glob(os.path.join(split_dir, "**", ext), recursive=True))
            
        class_counts = Counter()
        for img_path in images:
            # The class folder is the direct parent of the image file
            class_name = os.path.basename(os.path.dirname(img_path))
            class_counts[class_name] += 1
            
        print(f"\nSplit: {split} (Total Images: {len(images)})")
        for cls, count in sorted(class_counts.items()):
            print(f"  - {cls}: {count}")
    else:
        print(f"\nSplit: {split} - Directory not found: {split_dir}")


In [ ]:
class DeepPermNetDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths
        # Transform the image into a standard array/tensor first
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        img = Image.open(img_path).convert("RGB")
        tensor_img = self.transform(img) # Shape: (3, 192, 192)
        
        # Slicing: unfold into 9 distinct 64x64 tiles
        c, h, w = tensor_img.shape
        patch_h, patch_w = h // 3, w // 3
        # Unfold creates shape (3, 3, 3, 64, 64) -> rearrange to (9, 3, 64, 64)
        patches = tensor_img.unfold(1, patch_h, patch_h).unfold(2, patch_w, patch_w)
        patches = patches.contiguous().view(c, -1, patch_h, patch_w).permute(1, 0, 2, 3)
        
        # The Shuffler: Generate a completely random sequence
        target_perm = torch.randperm(9)
        shuffled_patches = patches[target_perm]
        
        # We need the inverse permutation as the target (where each shuffled patch belongs)
        inverse_perm = torch.argsort(target_perm)
        
        # Target aligns with PyTorch CrossEntropy 1D index vector
        return shuffled_patches, inverse_perm

class PuzzleDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=32):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size

    def setup(self, stage=None):
        def get_images(split_name):
            path = os.path.join(self.data_dir, split_name)
            files = []
            if os.path.exists(path):
                for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
                    files.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))
            return files

        self.train_files = get_images("train")
        self.val_files = get_images("valid")
        self.test_files = get_images("test")

        print(f"DataModule Setup: Found {len(self.train_files)} train, {len(self.val_files)} val, {len(self.test_files)} test images.")

        self.train_ds = DeepPermNetDataset(self.train_files)
        self.val_ds = DeepPermNetDataset(self.val_files)
        self.test_ds = DeepPermNetDataset(self.test_files)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=2)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=2)


In [ ]:
def sinkhorn_operator(log_alpha, n_iters=20, temp=0.1):
    log_alpha = log_alpha / temp
    for _ in range(n_iters):
        log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=-1, keepdim=True)
        log_alpha = log_alpha - torch.logsumexp(log_alpha, dim=-2, keepdim=True)
    return torch.exp(log_alpha)

def permutation_loss(y_hat, y, num_classes=9, displacement_weight=2.0):
    # Cross entropy using the log probabilities
    log_probs = torch.log(y_hat + 1e-8)
    ce_loss = F.nll_loss(log_probs.view(-1, num_classes), y.view(-1))
    
    # Displacement Loss
    positions = torch.arange(num_classes, device=y_hat.device).float()
    expected_pos = (y_hat * positions.view(1, 1, -1)).sum(dim=-1)
    displacement = (expected_pos - y.float()).abs()
    displacement_loss = (displacement ** 2).mean()
    
    return ce_loss + (displacement_weight * displacement_loss)

class DeepPermNetViT(pl.LightningModule):
    def __init__(self, learning_rate=1e-4, d_model=512, nhead=8, num_layers=4):
        super().__init__()
        self.save_hyperparameters()
        
        # Shared CNN Backbone
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten() # Outputs 2048 per patch (128 * 4 * 4)
        )
        
        self.token_projection = nn.Linear(2048, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.prediction_head = nn.Linear(d_model, 9)

    def forward(self, x):
        # x is shape (B, 9, 3, 64, 64)
        b, s, c, h, w = x.size()
        # Process patches individually then reshape back
        feats = self.feature_extractor(x.view(-1, c, h, w))
        tokens = self.token_projection(feats.view(b, s, -1))
        out = self.transformer(tokens)
        logits = self.prediction_head(out)
        return sinkhorn_operator(logits)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = permutation_loss(y_hat, y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = permutation_loss(y_hat, y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


In [ ]:
# Initialize DataModule and Model
datamodule = PuzzleDataModule(LOCAL_DATA_DIR, batch_size=BATCH_SIZE)
model = DeepPermNetViT(learning_rate=LR)

# Wandb Logger Setup (Allows resuming with an ID)
wandb.login()
logger = WandbLogger(project=PROJECT_NAME, log_model="all")

# Checkpointing Setup: Saves to 03_artifacts/models
checkpoint_callback = ModelCheckpoint(
    dirpath=MODEL_SAVE_DIR,
    filename='deep_perm_net-{epoch:02d}-{val_loss:.2f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min'
)

# Trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    logger=logger,
    callbacks=[checkpoint_callback],
    accelerator='auto',
    devices=1
)

# Resume from checkpoint if it exists in the artifacts folder
latest_ckpt = None
ckpts = glob.glob(os.path.join(MODEL_SAVE_DIR, "*.ckpt"))
if ckpts:
    latest_ckpt = max(ckpts, key=os.path.getctime)
    print(f"Resuming from checkpoint: {latest_ckpt}")

trainer.fit(model, datamodule=datamodule, ckpt_path=latest_ckpt)


In [ ]:
# Evaluate the model on test set and save weights as .pth
model.eval()

test_loader = datamodule.test_dataloader()
total_patches_correct = 0
total_puzzles_correct = 0
total_samples = 0

print("Evaluating on test set...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    for batch in test_loader:
        x, y = batch
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        
        # Predicted permutation
        preds = y_hat.argmax(dim=-1) # (B, 9)
        
        # Calculate correct patches & perfect puzzles
        correct_patches = (preds == y).sum().item()
        perfect_puzzles = (preds == y).all(dim=-1).sum().item()
        
        total_patches_correct += correct_patches
        total_puzzles_correct += perfect_puzzles
        total_samples += x.size(0)

if total_samples > 0:
    patch_acc = (total_patches_correct / (total_samples * 9)) * 100
    puzzle_acc = (total_puzzles_correct / total_samples) * 100
    print(f"\nEvaluation Results:")
    print(f"  Total Test Samples: {total_samples}")
    print(f"  Patch Placement Accuracy: {patch_acc:.2f}%")
    print(f"  Perfect Jigsaw Solve Accuracy: {puzzle_acc:.2f}%")
    
    # Log to wandb
    wandb.log({
        "test/patch_accuracy": patch_acc,
        "test/puzzle_accuracy": puzzle_acc
    })
else:
    print("No test samples available for evaluation.")

# Save PyTorch Model weights (.pth)
pth_path = os.path.join(MODEL_SAVE_DIR, "deep_perm_net.pth")
torch.save(model.state_dict(), pth_path)
print(f"\nSaved PyTorch model weights to: {pth_path}")

# Log the .pth file as a W&B Artifact
artifact = wandb.Artifact('deep-perm-net-weights', type='model')
artifact.add_file(pth_path)
wandb.log_artifact(artifact)

wandb.finish()


In [ ]:
# Inference Visualizer: Load a test image, shuffle patches, let the model solve it, and visualize
def visualize_puzzle_solving(model, datamodule, num_visualizations=3):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    test_files = datamodule.test_files
    if not test_files:
        print("No test files available for visualization.")
        return
        
    sampled_paths = random.sample(test_files, min(num_visualizations, len(test_files)))
    
    # Standard transform for visualization
    orig_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE))
    ])
    
    dataset = datamodule.test_ds
    
    for idx, path in enumerate(sampled_paths):
        try:
            ds_idx = test_files.index(path)
        except ValueError:
            continue
            
        shuffled_patches, inverse_perm = dataset[ds_idx]
        
        # Run inference
        input_tensor = shuffled_patches.unsqueeze(0).to(device) # (1, 9, 3, 64, 64)
        with torch.no_grad():
            y_hat = model(input_tensor) # (1, 9, 9)
            pred_inv_perm = y_hat.squeeze(0).argmax(dim=-1).cpu() # (9,)
            
        # Reconstruct the image from shuffled patches
        reconstructed_patches = [None] * 9
        for i in range(9):
            pos = pred_inv_perm[i].item()
            reconstructed_patches[pos] = shuffled_patches[i]
            
        for i in range(9):
            if reconstructed_patches[i] is None:
                reconstructed_patches[i] = torch.zeros(3, 64, 64)
                
        # Reassemble 3x3 images
        shuffled_grid = torch.zeros(3, 192, 192)
        for i in range(9):
            r, c = i // 3, i % 3
            shuffled_grid[:, r*64:(r+1)*64, c*64:(c+1)*64] = shuffled_patches[i]
            
        recon_grid = torch.zeros(3, 192, 192)
        for i in range(9):
            r, c = i // 3, i % 3
            recon_grid[:, r*64:(r+1)*64, c*64:(c+1)*64] = reconstructed_patches[i]
            
        # Plot
        orig_img = Image.open(path).convert("RGB")
        orig_img = orig_transform(orig_img)
        
        shuffled_np = shuffled_grid.permute(1, 2, 0).numpy()
        recon_np = recon_grid.permute(1, 2, 0).clip(0, 1).numpy()
        
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(orig_img)
        axes[0].set_title(f"Original: {os.path.basename(path)}")
        axes[0].axis('off')
        
        axes[1].imshow(shuffled_np)
        axes[1].set_title("Shuffled Jigsaw")
        axes[1].axis('off')
        
        axes[2].imshow(recon_np)
        axes[2].set_title("Model Reconstruction")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()

# Run visualizer
try:
    visualize_puzzle_solving(model, datamodule)
except Exception as e:
    print(f"Could not run visualization: {e}")
